### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installation

In [2]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Gemma-3!
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install msgspec
!pip install blake3
!pip install gguf
!pip install networkx==2.8.8 --force-reinstall

In [3]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    # Force reinstall Pillow to fix the _imaging compiled extension error

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [ ]:
!pip install --force-reinstall pillow==11.3.0
# Force restart the kernel session to load the correct library version
import os
os.kill(os.getpid(), 9)

  Using cached pillow-11.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.0 kB)
Using cached pillow-11.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.6 MB)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.34.0 requires fastsafetensors>=0.3.3, which is not installed.
vllm 0.34.0 requires flashinfer-python==0.7.3, which is not installed.
vllm 0.34.0 requires humming-kernels[cu13]==0.1.19, which is not installed.
vllm 0.34.0 requires instanttensor>=0.2.2, which is not installed.
vllm 0.34.0 requires nvidia-cudnn-frontend>=1.31.0, which is not installed.
vllm 0.34.0 requires nvidia-cutlass-dsl[cu13]==4.8.0, which is not installed.
vllm 0.34.0 requires PyNvVideo

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [1]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1121 05:07:11.479000 6728 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1121 05:07:11.533000 6728 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1121 05:07:14.527000 6728 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1121 05:07:14.529000 6728 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1121 05:07:14.532000 6728 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.9: Fast Gemma3 patching. Transformers: 5.18.0. vLLM: 0.34.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [ ]:
import io
import pandas as pd
from datasets import Dataset, DatasetDict

file_path = "/content/dataasilida_cleaned.csv"

In [ ]:
# 1. Fix the quoted header on line 1
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

lines[0] = "input,output\n"
df = pd.read_csv(io.StringIO("".join(lines)))

# 2. Split the 7 rows where question & answer were combined in one column
mask = df["output"].isna()
split_cols = df.loc[mask, "input"].str.split(r"(?<=\?)\s*,?\s*", n=1, expand=True)
df.loc[mask, "input"] = split_cols[0].str.strip()
df.loc[mask, "output"] = split_cols[1].str.strip()

# 3. Convert to a Hugging Face DatasetDict
raw_dataset = Dataset.from_pandas(df, preserve_index=False)

# 4. Perform a Train/Validation split (85% train, 15% validation)
dataset = raw_dataset.train_test_split(test_size=0.15, seed=3407)

print(f"Training set size: {len(dataset['train'])}")
print(f"Validation set size: {len(dataset['test'])}")
print("Sample training prompt:", dataset["train"]["input"][0])

Training set size: 442
Validation set size: 79
Sample training prompt: តើ​រូបិយប័ណ្ណ​អ្វីខ្លះ ដែល​អាច​ធ្វើប្រតិបត្តិការ​តាម​សេវាអេស៊ីលីដាម៉ូបាល បាន


In [ ]:
def formatting_prompts_func(examples):
    # Always parse inputs and outputs as lists to process uniformly
    is_batched = isinstance(examples["input"], list)
    instructions = examples["input"] if is_batched else [examples["input"]]
    outputs = examples["output"] if is_batched else [examples["output"]]

    texts = []
    for instruction, output in zip(instructions, outputs):
        messages = [
            {
                "role": "user", 'content': str(instruction).strip(" ")
            },
            {
                "role": "assistant", 'content': str(output)
            }
        ]
        text = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)
        texts.append(text)

    # Always return a plain list of strings as required by Unsloth SFTTrainer
    return texts

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/521 [00:00<?, ? examples/s]

In [ ]:
dataset['train']['input'][0]

'តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី'

In [ ]:
dataset['train']['text'][0]

'<bos><start_of_turn>user\nតើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី<end_of_turn>\n<start_of_turn>model\nប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សាយដោយ ធនាគារ អេស៊ីលីដា ភីអិលស៊ី សម្រាប់ផ្តល់ជូនអតិថិជនក្នុងការធ្វើប្រតិបត្តិការតាមរយៈ E-Commerce ម៉ាស៊ីនឆូតកាត (POS) ម៉ាស៊ីនអេធីអឹម (ATM) ដែលមានបិទស្លាកសញ្ញា ចេស៊ីប៊ី នៅទូទាំងប្រទេស និងពាសពេញពិភពលោក ដោយធ្វើការទូទាត់សាច់ប្រាក់ចេញពីគណនីរបស់ម្ចាស់ប័ណ្ណដែលមាននៅធនាគារ អេស៊ីលីដា។<end_of_turn>\n'

Let's see how row 100 looks like!

In [ ]:
dataset['train']['input'][1]

'តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ផ្តល់នូវអត្ថប្រយោជន៍អ្វីខ្លះ'

In [ ]:
dataset['train']['input'][100]

'តើត្រូវមានលក្ខខណ្ឌតម្រូវអ្វីខ្លះ សម្រាប់ការស្នើសុំបង្កើតប័ណ្ណនិម្មិតតាមសេវាអេស៊ីលីដាម៉ូបាល '

Let's see how the chat template did! Notice `Gemma-3` default adds a `<bos>`!

In [ ]:
dataset['train']['output'][00]

'ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សាយដោយ ធនាគារ អេស៊ីលីដា ភីអិលស៊ី សម្រាប់ផ្តល់ជូនអតិថិជនក្នុងការធ្វើប្រតិបត្តិការតាមរយៈ E-Commerce ម៉ាស៊ីនឆូតកាត (POS) ម៉ាស៊ីនអេធីអឹម (ATM) ដែលមានបិទស្លាកសញ្ញា ចេស៊ីប៊ី នៅទូទាំងប្រទេស និងពាសពេញពិភពលោក ដោយធ្វើការទូទាត់សាច់ប្រាក់ចេញពីគណនីរបស់ម្ចាស់ប័ណ្ណដែលមាននៅធនាគារ អេស៊ីលីដា។'

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
# from trl import SFTTrainer, SFTConfig
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     eval_dataset = None, # Can set up evaluation!
#     args = SFTConfig(
#         dataset_text_field = "text",
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4, # Use GA to mimic batch size!
#         warmup_steps = 5,
#         # num_train_epochs = 1, # Set this for 1 full training run.
#         max_steps = 60,
#         learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
#         logging_steps = 1,
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         report_to = "none", # Use this for WandB etc
#     ),
# )

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],       # Added validation dataset
    formatting_func = formatting_prompts_func,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,    # Added evaluation batch size
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        eval_strategy = "steps",            # Enable evaluation metrics logging
        eval_steps = 10,                   # Calculate validation loss every 10 steps
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/442 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/79 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map:   0%|          | 0/521 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again:

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><start_of_turn>user\nតើត្រូវមានលក្ខខណ្ឌតម្រូវអ្វីខ្លះ សម្រាប់ការស្នើសុំបង្កើតប័ណ្ណនិម្មិតតាមសេវាអេស៊ីលីដាម៉ូបាល<end_of_turn>\n<start_of_turn>model\nអតិថិជនត្រូវមានគណនីប្រភព (គណនីធនាគារ) ជាប្រាក់ដុល្លារអាមេរិក ដែលបានភ្ជាប់ជាមួយសេវាអេ-ស៊ីលីដាម៉ូបាល។ \\n ត្រូវបម្រុងសមតុល្យគណនីឲ្យបានគ្រប់គ្រាន់ រាល់ពេលបង្កើតប័ណ្ណ និងពេលទូទាត់ថ្លៃទំនិញ-សេវាកម្ម ។<end_of_turn>\n'

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                  អតិថិជនត្រូវមានគណនីប្រភព (គណនីធនាគារ) ជាប្រាក់ដុល្លារអាមេរិក ដែលបានភ្ជាប់ជាមួយសេវាអេ-ស៊ីលីដាម៉ូបាល។ \\n ត្រូវបម្រុងសមតុល្យគណនីឲ្យបានគ្រប់គ្រាន់ រាល់ពេលបង្កើតប័ណ្ណ និងពេលទូទាត់ថ្លៃទំនិញ-សេវាកម្ម ។<end_of_turn> '

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
4.518 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 442 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)


Step,Training Loss,Validation Loss
10,1.966068,2.551219
20,2.033857,1.923410
30,1.733828,1.635275
40,1.414769,1.549926
50,1.447842,1.498034
60,1.297629,1.479364


Unsloth: Not an error, but `num_items_in_batch` could not be counted for Gemma3ForConditionalGeneration's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


### Training & Validation Loss Analysis (Experiment 1: ACLEDA Khmer FAQ)

| Step | Training Loss | Validation Loss |
| :--- | :---: | :---: |
| 10 | 1.9661 | 2.5512 |
| 20 | 2.0339 | 1.9234 |
| 30 | 1.7338 | 1.6353 |
| 40 | 1.4148 | 1.5499 |
| 50 | 1.4478 | 1.4980 |
| 60 | 1.2976 | 1.4794 |

**Diagnosis: Good Fit (Healthy Learning)**
* **No Overfitting:** Both Training Loss (`1.97 -> 1.30`) and Validation Loss (`2.55 -> 1.48`) decrease steadily together across all 60 steps. Validation loss never spikes upward while training loss drops, confirming the model is generalizing rather than memorizing.
* **No Underfitting:** Both losses drop significantly from their initial values (~42% reduction in validation loss), showing the learning rate (`2e-4`) and LoRA rank (`r=8`) effectively adapted Gemma 3 to the Khmer banking domain.

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

358.8583 seconds used for training.
5.98 minutes used for training.
Peak reserved memory = 5.08 GB.
Peak reserved memory for training = 0.562 GB.
Peak reserved memory % of max memory = 34.883 %.
Peak reserved memory for training % of max memory = 3.859 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "អ្វីជាអេសុីលីដា?",
    }]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)
outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nអ្វីជាអេសុីលីដា?<end_of_turn>\n<start_of_turn>model\nអេសុីលីដា (ASG) គឺជាលក្ខណ្ឌមួយដែលត្រូវបានអនុម័តដោយសម្តាអន្តិក ឬអាជ្ញាធរិយាគណៈចៅការកម្លាំង និង ភ្នាក់ងារ អាជ្ញាធរិយា']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "ទៅមុខ១០ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

មើលទៅសេវាទាំងនេះដែលធ្វើទៅបាន និងពាក់ព័ន្ធនឹងចំណុចបញ្ជាក់ពីខាងជើង ។ ត្រាំចាំង ព្រៃសិលា អៀនវ៉ាន់ ចា រីន ចាយ រីន មានទីកន្លែងបិទសិ


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving

Unsloth: Restored added_tokens_decoder metadata in gemma-3/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3.


['gemma-3/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "បន្តទៅក្រោយ៩ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

អាស័យភ្នាក់បត្រប័ណ្ណ អតិថិជន របស់លោក/លោកស្រី នាងខ្ញុំ ដូចជា អិលាយ ប៉ាត់ហ៊ីវ និង សហករណ៍ ព្រហោង នៃ ហោរលក្ខខណ្ឌ របស់ ខ្ញុំ ប៉


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
        token = "hf_...",
    )

## Run Experiment 2 on a Different Dataset

Free GPU Memory & Load a Fresh Base Model

In [3]:
import gc, torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

# Safely delete existing models/trainers if they exist in memory
for var in ['model', 'trainer', 'trainer_exp2']:
    if var in globals():
        del globals()[var]

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

==((====))==  Unsloth 2026.11.9: Fast Gemma3 patching. Transformers: 5.18.0. vLLM: 0.34.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

In [4]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

Load & Format the New Dataset (gbharti/finance-alpaca)

In [5]:
from datasets import load_dataset

# Load 521 samples to match Experiment 1 size for a fair comparison
raw_exp2 = load_dataset("gbharti/finance-alpaca", split="train[:521]")
dataset_exp2 = raw_exp2.train_test_split(test_size=0.15, seed=3407)

print(f"Exp 2 Train size: {len(dataset_exp2['train'])}")
print(f"Exp 2 Val size: {len(dataset_exp2['test'])}")

README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json: reconstructing file:   0%|          |  0.00B / 42.9MB            

Cleaned_date.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

Exp 2 Train size: 442
Exp 2 Val size: 79


In [12]:
def formatting_prompts_exp2(examples):
    is_batched = isinstance(examples["instruction"], list)
    instructions = examples["instruction"] if is_batched else [examples["instruction"]]
    inputs = examples["input"] if is_batched else [examples["input"]]
    outputs = examples["output"] if is_batched else [examples["output"]]

    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        user_msg = f"{inst}\n{inp}".strip() if inp and str(inp).strip() else str(inst).strip()
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": str(out).strip()},
        ]
        text = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)
        texts.append(text)
    return {"text": texts}

Train & Run Quick Inference Test

In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

# Map the formatting function to dataset_exp2 to add the 'text' field
dataset_exp2 = dataset_exp2.map(formatting_prompts_exp2, batched = True)

trainer_exp2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_exp2["train"],
    eval_dataset = dataset_exp2["test"],
    dataset_text_field = "text",
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_exp2",
        report_to = "none",
    ),
)

trainer_exp2 = train_on_responses_only(
    trainer_exp2,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map:   0%|          | 0/442 [00:00<?, ? examples/s]

Map:   0%|          | 0/79 [00:00<?, ? examples/s]

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/442 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/79 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/442 [00:00<?, ? examples/s]

Map:   0%|          | 0/79 [00:00<?, ? examples/s]

In [14]:
trainer_stats_exp2 = trainer_exp2.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 106}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 442 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
10,2.593525,2.749279
20,2.397758,2.584712
30,2.584175,2.549343
40,2.407015,2.536113
50,2.468073,2.528084
60,2.392755,2.525307


Filter:   0%|          | 0/79 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Gemma3ForConditionalGeneration's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs_exp2/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs_exp2/checkpoint-60.


In [15]:
# Quick Inference Test on Finance Question
messages = [{"role": "user", "content": [{"type": "text", "text": "What is the difference between a debit card and a credit card?"}]}]
text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
outputs = model.generate(
    **tokenizer([text], return_tensors="pt").to("cuda"),
    max_new_tokens=100,
    temperature=1.0, top_p=0.95, top_k=64,
)
print(tokenizer.batch_decode(outputs)[0])

<bos><start_of_turn>user
What is the difference between a debit card and a credit card?<end_of_turn>
<start_of_turn>model
The important difference is that credit cards **borrow** money from the bank to make the purchases for you, as if you had money in your savings account.  Debit cards on the other hand **take** money directly out of your checking account.  If you are comfortable spending money you don't have (i.e., you're putting it on a credit card), then credit cards can be a great tool as long as you're smart about it and always pay it back.  


### Final Comparison Markdown

### Experiment Comparison: Khmer Banking FAQ vs. English Finance QA

| Metric / Aspect | Experiment 1 (ACLEDA Khmer FAQ) | Experiment 2 (Finance-Alpaca English) |
| :--- | :--- | :--- |
| **Base Model & LoRA** | `gemma-3-4b-it` (`r=8`, `alpha=8`) | `gemma-3-4b-it` (`r=8`, `alpha=8`) |
| **Dataset Size** | 521 rows (442 train / 79 val) | 521 rows (442 train / 79 val) |
| **Steps / Learning Rate** | 60 steps / `2e-4` | 60 steps / `2e-4` |
| **Final Training Loss (Step 60)** | `1.2976` | `2.3927` |
| **Final Validation Loss (Step 60)** | `1.4794` | `2.5253` |
| **Fit Diagnosis** | Good fit (steady decrease) | Good fit (steady decrease) |

**Key Observations:**
1. **Language & Pretraining Alignment:** Gemma 3 has much stronger English pretraining than Khmer, which typically results in lower initial/final loss and more fluent, detailed responses on the English finance dataset compared to specialized Khmer banking terminology.
2. **Convergence Behavior:** With identical hyperparameters (`max_steps=60`, `lr=2e-4`, `r=8`), both runs converge smoothly without overfitting, proving that LoRA fine-tuning effectively adapts the model across different languages and financial domains using minimal trainable parameters (0.35%).